# Cardiac Analyst — PTB-XL 12-Lead ECG Classification
### 1D Residual Convolutional Neural Network (ResNet-1D) for Multi-Label Cardiac Diagnostics

> **⚠️ Regulatory Notice**: *Cardiac Analyst is a research and educational prototype only. It is NOT a certified medical device and must never be used for clinical diagnosis, patient triage, or direct medical decision-making.*

---

### Project Overview
This notebook implements an end-to-end deep learning pipeline for multi-label classification of 12-lead ECGs using the **PhysioNet PTB-XL v1.0.3** dataset.
The model predicts five diagnostic superclasses:
1. **NORM**: Normal ECG
2. **MI**: Myocardial Infarction
3. **STTC**: ST/T-Wave Changes
4. **CD**: Conduction Disturbance
5. **HYP**: Hypertrophy

### Methodological Principles
- **Strict Data Splitting**: Official PhysioNet Folds 1–8 for Training, Fold 9 for Validation, and Fold 10 for Hold-Out Testing.
- **Leakage Prevention**: Per-class classification thresholds are optimized strictly on Fold 9 validation data and never on Fold 10.
- **Single-Pass Test Evaluation**: Fold 10 is evaluated exactly once after training and threshold tuning are locked.
- **Deployment Export**: Seamless export to ONNX format for lightweight CPU inference in FastAPI/web microservices.


In [ ]:
# ---------------------------------------------------------------------------
# 1. Environment Setup & Dependency Installation
# ---------------------------------------------------------------------------
import subprocess
import sys
import os

# Install wfdb and onnx conversion dependencies if missing
required_packages = ["wfdb", "onnx", "onnxruntime", "tf2onnx", "seaborn"]
for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing missing package: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import ast
import time
import math
import gc
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb

import tensorflow as tf
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix, classification_report
)

print(f"Python version     : {sys.version.split()[0]}")
print(f"TensorFlow version : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Detected      : {len(gpus)}")
for g in gpus:
    print(f"  - {g}")
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print(f"    Memory growth note: {e}")


In [ ]:
# ---------------------------------------------------------------------------
# 2. Pipeline Constants & Hyperparameters
# ---------------------------------------------------------------------------
DIAGNOSTIC_SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
SUPERCLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(DIAGNOSTIC_SUPERCLASSES)}
NUM_CLASSES = len(DIAGNOSTIC_SUPERCLASSES)

STANDARD_LEAD_ORDER = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
SAMPLING_RATE = 100        # Standard 100 Hz low-resolution PTB-XL records
SIGNAL_LENGTH = 1000       # 10 seconds * 100 samples/sec = 1000 samples
NUM_LEADS = 12
INPUT_SHAPE = (SIGNAL_LENGTH, NUM_LEADS)

# PhysioNet PTB-XL Official Fold Partitions
TRAIN_FOLDS = list(range(1, 9))  # Folds 1-8 (~80%)
VAL_FOLDS = [9]                  # Fold 9 (~10%)
TEST_FOLDS = [10]                # Fold 10 (~10% Hold-out)

# Training Hyperparameters
BATCH_SIZE = 64
EPOCHS = 20
INITIAL_LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = "./output_artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Artifacts output directory: {OUTPUT_DIR}")


## 3. Dataset Discovery & Path Resolution
The cell below automatically scans for PTB-XL dataset paths across multiple environments:
- **Kaggle**: `/kaggle/input/ptb-xl-1-0-3/...`
- **Google Colab / Local**: Local directory or automatic download of official PhysioNet v1.0.3 archive.


In [ ]:
def locate_or_download_ptbxl():
    candidate_paths = [
        "/kaggle/input/ptb-xl-1-0-3/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3",
        "/kaggle/input/ptb-xl-1-0-3",
        "/kaggle/input/datasets/garethwmch/ptb-xl-1-0-3/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3",
        "/kaggle/input/ptb-xl-dataset",
        "./ptb-xl-1.0.3",
        "./scratch/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3",
        "./ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
    ]
    for path in candidate_paths:
        csv_check = os.path.join(path, "ptbxl_database.csv")
        if os.path.exists(csv_check):
            print(f"✓ Found PTB-XL dataset at: {path}")
            return os.path.abspath(path)

    # If running in Colab/local without dataset, download PhysioNet release
    print("PTB-XL not found locally. Downloading official PhysioNet v1.0.3 (~1.8 GB)...")
    import urllib.request
    import zipfile
    url = "https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
    dest_zip = "./ptb-xl-1.0.3.zip"
    urllib.request.urlretrieve(url, dest_zip)
    print("Extracting zip archive...")
    with zipfile.ZipFile(dest_zip, 'r') as zip_ref:
        zip_ref.extractall("./ptb-xl-1.0.3")
    return os.path.abspath("./ptb-xl-1.0.3/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")

DATA_DIR = locate_or_download_ptbxl()
print(f"Working Data Directory: {DATA_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# 4. Data Parsing & Diagnostic Superclass Aggregation
# ---------------------------------------------------------------------------
def load_and_preprocess_metadata(data_dir: str):
    db_csv = os.path.join(data_dir, "ptbxl_database.csv")
    scp_csv = os.path.join(data_dir, "scp_statements.csv")

    df = pd.read_csv(db_csv, index_col="ecg_id")
    scp_df = pd.read_csv(scp_csv, index_col=0)

    # Filter scp_statements to diagnostic statements only
    diag_scp = scp_df[scp_df["diagnostic"] == 1.0]

    # Convert string representation of scp_codes dict to actual dict
    df["scp_codes_dict"] = df["scp_codes"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    # Map individual SCP statement keys to the 5 diagnostic superclasses
    def get_superclasses(codes_dict):
        classes = set()
        for code in codes_dict.keys():
            if code in diag_scp.index:
                super_cls = diag_scp.loc[code, "diagnostic_class"]
                if pd.notna(super_cls) and super_cls in SUPERCLASS_TO_IDX:
                    classes.add(super_cls)
        return list(classes)

    df["diagnostic_superclass"] = df["scp_codes_dict"].apply(get_superclasses)

    # Keep records having at least 1 diagnostic superclass
    clean_df = df[df["diagnostic_superclass"].apply(lambda x: len(x) > 0)].copy()

    # Construct one-hot multi-label binary array (N, 5)
    labels = np.zeros((len(clean_df), NUM_CLASSES), dtype=np.float32)
    for idx, (_, row) in enumerate(clean_df.iterrows()):
        for cls_name in row["diagnostic_superclass"]:
            labels[idx, SUPERCLASS_TO_IDX[cls_name]] = 1.0

    clean_df["label_vector"] = list(labels)

    print(f"Total Database Records : {len(df)}")
    print(f"Clean Diagnostic Records: {len(clean_df)}")
    return clean_df

metadata_df = load_and_preprocess_metadata(DATA_DIR)
metadata_df[["strat_fold", "age", "sex", "diagnostic_superclass"]].head()


In [ ]:
# Inspect partition distribution across Folds
train_df = metadata_df[metadata_df["strat_fold"].isin(TRAIN_FOLDS)].copy()
val_df   = metadata_df[metadata_df["strat_fold"].isin(VAL_FOLDS)].copy()
test_df  = metadata_df[metadata_df["strat_fold"].isin(TEST_FOLDS)].copy()

print(f"Train Records (Folds 1-8) : {len(train_df)}")
print(f"Val Records   (Fold 9)    : {len(val_df)}")
print(f"Test Records  (Fold 10)   : {len(test_df)}")

# Superclass distribution
counts = {cls: sum(df["diagnostic_superclass"].apply(lambda cl: cls in cl)) for cls in DIAGNOSTIC_SUPERCLASSES for df in [metadata_df]}
plt.figure(figsize=(8, 4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()), palette="viridis")
plt.title("PTB-XL Diagnostic Superclass Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Superclass")
plt.ylabel("Record Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 5. Signal Loading & 12-Lead ECG Visualizer
# ---------------------------------------------------------------------------
def load_single_wfdb_signal(record_path_no_ext: str, target_length: int = 1000) -> np.ndarray:
    signal, fields = wfdb.rdsamp(record_path_no_ext)
    sig_len, num_leads = signal.shape

    # Pad or truncate to target_length
    if sig_len < target_length:
        pad = np.zeros((target_length - sig_len, num_leads), dtype=signal.dtype)
        signal = np.vstack([signal, pad])
    elif sig_len > target_length:
        signal = signal[:target_length, :]

    # Standardize lead ordering
    fields_leads = [lead.upper() for lead in fields["sig_name"]]
    lead_indices = []
    for target_lead in STANDARD_LEAD_ORDER:
        upper_target = target_lead.upper()
        if upper_target in fields_leads:
            lead_indices.append(fields_leads.index(upper_target))
        else:
            lead_indices.append(0)

    signal = signal[:, lead_indices]
    signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return signal

def plot_12_lead_ecg(signal: np.ndarray, title: str = "12-Lead ECG (10 Seconds @ 100 Hz)"):
    """Plots standard 12-lead layout (3x4 grid) in millivolts."""
    fig, axes = plt.subplots(6, 2, figsize=(14, 10), sharex=True)
    axes = axes.flatten()
    time_axis = np.linspace(0, 10, signal.shape[0])

    for i, lead_name in enumerate(STANDARD_LEAD_ORDER):
        ax = axes[i]
        ax.plot(time_axis, signal[:, i], color='#0284c7', linewidth=1.1)
        ax.set_ylabel(f"{lead_name} (mV)", fontsize=9, fontweight='bold')
        ax.grid(True, linestyle=':', alpha=0.6)
        if i >= 10:
            ax.set_xlabel("Time (seconds)", fontsize=9)

    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

# Visualize an example Normal vs MI record
sample_row = metadata_df[metadata_df["diagnostic_superclass"].apply(lambda x: "MI" in x)].iloc[0]
sample_path = os.path.join(DATA_DIR, sample_row["filename_lr"])
sample_sig = load_single_wfdb_signal(sample_path)
plot_12_lead_ecg(sample_sig, f"Sample 12-Lead ECG: Record {sample_row.name} (Classes: {sample_row['diagnostic_superclass']})")


In [ ]:
# ---------------------------------------------------------------------------
# 6. High-Throughput In-Memory Signal Ingestion
# ---------------------------------------------------------------------------
def load_split_signals(df: pd.DataFrame, data_dir: str, max_workers: int = 8) -> np.ndarray:
    print(f"Loading {len(df)} records with {max_workers} worker threads...")
    file_paths = [os.path.join(data_dir, f_lr) for f_lr in df["filename_lr"]]
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        signals = list(executor.map(load_single_wfdb_signal, file_paths))
    return np.stack(signals, axis=0)

X_train = load_split_signals(train_df, DATA_DIR)
y_train = np.stack(train_df["label_vector"].values, axis=0)

X_val = load_split_signals(val_df, DATA_DIR)
y_val = np.stack(val_df["label_vector"].values, axis=0)

X_test = load_split_signals(test_df, DATA_DIR)
y_test = np.stack(test_df["label_vector"].values, axis=0)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val   shape: {X_val.shape}, y_val   shape: {y_val.shape}")
print(f"X_test  shape: {X_test.shape}, y_test  shape: {y_test.shape}")


In [ ]:
# ---------------------------------------------------------------------------
# 7. Training Augmentations & Keras Sequence
# ---------------------------------------------------------------------------
def augment_signal_batch(x_batch: np.ndarray) -> np.ndarray:
    """Physiological augmentations applied ONLY during training:
    - Amplitude jitter (±5% electrode scaling)
    - Small cyclic time shifts (±20 samples / 0.2s)
    - Gaussian baseline noise
    """
    augmented = x_batch.copy()
    b_size = len(x_batch)

    # 1. Amplitude scale
    scales = np.random.uniform(0.95, 1.05, size=(b_size, 1, 1)).astype(np.float32)
    augmented = augmented * scales

    # 2. Circular time shifts
    shifts = np.random.randint(-20, 21, size=b_size)
    for i in range(b_size):
        if shifts[i] != 0:
            augmented[i] = np.roll(augmented[i], shifts[i], axis=0)

    # 3. Small Gaussian noise
    noise = np.random.normal(0.0, 0.005, size=augmented.shape).astype(np.float32)
    augmented = augmented + noise
    return augmented

class ECGSequence(tf.keras.utils.Sequence):
    def __init__(self, x_data: np.ndarray, y_data: np.ndarray, batch_size: int = 64, is_training: bool = False):
        self.x_data = x_data
        self.y_data = y_data
        self.batch_size = batch_size
        self.is_training = is_training
        self.indices = np.arange(len(self.x_data))
        if self.is_training:
            np.random.shuffle(self.indices)

    def __len__(self):
        return math.ceil(len(self.x_data) / self.batch_size)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        x_batch = self.x_data[batch_idx]
        y_batch = self.y_data[batch_idx]
        if self.is_training:
            x_batch = augment_signal_batch(x_batch)
        return x_batch, y_batch

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.indices)

train_gen = ECGSequence(X_train, y_train, batch_size=BATCH_SIZE, is_training=True)
val_gen   = ECGSequence(X_val, y_val, batch_size=BATCH_SIZE, is_training=False)


In [ ]:
# ---------------------------------------------------------------------------
# 8. Residual 1D CNN Model Architecture
# ---------------------------------------------------------------------------
def make_residual_block_1d(x: tf.Tensor, filters: int, kernel_size: int = 7, stride: int = 1, name: str = "") -> tf.Tensor:
    shortcut = x
    if stride != 1 or x.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv1D(
            filters=filters, kernel_size=1, strides=stride, padding="same", use_bias=False, name=f"{name}_sc_conv"
        )(x)
        shortcut = tf.keras.layers.BatchNormalization(name=f"{name}_sc_bn")(shortcut)

    y = tf.keras.layers.Conv1D(
        filters=filters, kernel_size=kernel_size, strides=stride, padding="same", use_bias=False, name=f"{name}_conv1"
    )(x)
    y = tf.keras.layers.BatchNormalization(name=f"{name}_bn1")(y)
    y = tf.keras.layers.ReLU(name=f"{name}_relu1")(y)

    y = tf.keras.layers.Conv1D(
        filters=filters, kernel_size=kernel_size, strides=1, padding="same", use_bias=False, name=f"{name}_conv2"
    )(y)
    y = tf.keras.layers.BatchNormalization(name=f"{name}_bn2")(y)

    out = tf.keras.layers.Add(name=f"{name}_add")([shortcut, y])
    out = tf.keras.layers.ReLU(name=f"{name}_relu2")(out)
    return out

def build_resnet1d_keras(input_shape=(1000, 12), num_classes=5) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=input_shape, name="ecg_12lead_input")

    # Stem: Wide convolution (kernel 15) to capture P-QRS-T complexes
    x = tf.keras.layers.Conv1D(filters=64, kernel_size=15, strides=2, padding="same", use_bias=False, name="stem_conv")(inputs)
    x = tf.keras.layers.BatchNormalization(name="stem_bn")(x)
    x = tf.keras.layers.ReLU(name="stem_relu")(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=3, strides=2, padding="same", name="stem_pool")(x)

    # 4 Residual Stages (64 -> 128 -> 256 -> 512 channels)
    x = make_residual_block_1d(x, filters=64, kernel_size=7, stride=1, name="st1_b1")
    x = make_residual_block_1d(x, filters=64, kernel_size=7, stride=1, name="st1_b2")

    x = make_residual_block_1d(x, filters=128, kernel_size=7, stride=2, name="st2_b1")
    x = make_residual_block_1d(x, filters=128, kernel_size=7, stride=1, name="st2_b2")

    x = make_residual_block_1d(x, filters=256, kernel_size=7, stride=2, name="st3_b1")
    x = make_residual_block_1d(x, filters=256, kernel_size=7, stride=1, name="st3_b2")

    x = make_residual_block_1d(x, filters=512, kernel_size=7, stride=2, name="st4_b1")
    x = make_residual_block_1d(x, filters=512, kernel_size=7, stride=1, name="st4_b2")

    # Global Average Pooling + Dropout + Multi-label Sigmoid Head
    x = tf.keras.layers.GlobalAveragePooling1D(name="global_avg_pool")(x)
    x = tf.keras.layers.Dropout(0.3, name="head_dropout")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="sigmoid", name="class_probabilities")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name="ResNet1D_ECG")

model = build_resnet1d_keras(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES)
model.summary()


In [ ]:
# ---------------------------------------------------------------------------
# 9. Loss Function & Validation Callback
# ---------------------------------------------------------------------------
# Positive class weighting (N_neg / N_pos) to balance under-represented classes (HYP, MI)
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
pos_weights = (neg_counts / (pos_counts + 1e-5)).astype(np.float32)
pos_weights_tf = tf.constant(pos_weights, dtype=tf.float32)

print("Class Positive Weights:")
for cls_name, w in zip(DIAGNOSTIC_SUPERCLASSES, pos_weights):
    print(f"  - {cls_name:5s}: {w:.3f}")

def weighted_bce_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    loss = - (pos_weights_tf * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    return tf.reduce_mean(loss)

class ValidationAUROCCallback(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val, checkpoint_path):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.checkpoint_path = checkpoint_path
        self.best_auc = -1.0

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, batch_size=BATCH_SIZE, verbose=0)
        aucs = []
        for i in range(NUM_CLASSES):
            try:
                auc_c = roc_auc_score(self.y_val[:, i], preds[:, i])
            except ValueError:
                auc_c = 0.5
            aucs.append(auc_c)
        macro_auc = float(np.mean(aucs))
        print(f" -> Epoch {epoch + 1:02d} Val Macro-AUROC: {macro_auc:.4f} (Best: {self.best_auc:.4f})")
        if macro_auc > self.best_auc:
            self.best_auc = macro_auc
            self.model.save(self.checkpoint_path)
            print(f"    ⭐ Saved checkpoint to: {self.checkpoint_path}")

checkpoint_file = os.path.join(OUTPUT_DIR, "best_resnet1d_ptbxl.keras")
auc_callback = ValidationAUROCCallback(X_val, y_val, checkpoint_file)


In [ ]:
# ---------------------------------------------------------------------------
# 10. Model Compilation & Training Execution
# ---------------------------------------------------------------------------
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=INITIAL_LR, weight_decay=WEIGHT_DECAY),
    loss=weighted_bce_loss,
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")]
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[auc_callback, lr_scheduler, early_stop],
    verbose=1
)


In [ ]:
# ---------------------------------------------------------------------------
# 11. Per-Class Threshold Optimization (STRICTLY ON FOLD 9)
# ---------------------------------------------------------------------------
# Reload best checkpoint
best_model = tf.keras.models.load_model(
    checkpoint_file, custom_objects={"weighted_bce_loss": weighted_bce_loss}
)

val_preds = best_model.predict(X_val, batch_size=BATCH_SIZE)

optimal_thresholds = {}
print("Optimal F1-maximizing Thresholds computed on Validation (Fold 9):")
candidate_thresholds = np.linspace(0.10, 0.90, 81)

for idx, cls_name in enumerate(DIAGNOSTIC_SUPERCLASSES):
    y_true_c = y_val[:, idx]
    y_pred_c = val_preds[:, idx]
    best_f1 = -1.0
    best_th = 0.50
    for th in candidate_thresholds:
        f1 = f1_score(y_true_c, (y_pred_c >= th).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_th = th
    optimal_thresholds[cls_name] = round(float(best_th), 3)
    print(f"  - {cls_name:5s} : Threshold = {best_th:.3f} (Val F1 = {best_f1:.4f})")


In [ ]:
# ---------------------------------------------------------------------------
# 12. Final Hold-Out Test Evaluation (FOLD 10 - UNTOUCHED)
# ---------------------------------------------------------------------------
test_preds = best_model.predict(X_test, batch_size=BATCH_SIZE)

metrics_records = []
for idx, cls_name in enumerate(DIAGNOSTIC_SUPERCLASSES):
    y_true_c = y_test[:, idx]
    y_prob_c = test_preds[:, idx]
    th = optimal_thresholds[cls_name]
    y_bin_c = (y_prob_c >= th).astype(int)

    auc_c = roc_auc_score(y_true_c, y_prob_c)
    f1_c = f1_score(y_true_c, y_bin_c, zero_division=0)
    sens_c = recall_score(y_true_c, y_bin_c, zero_division=0)  # Sensitivity
    tn, fp, fn, tp = confusion_matrix(y_true_c, y_bin_c).ravel()
    spec_c = tn / (tn + fp) if (tn + fp) > 0 else 0.0          # Specificity

    metrics_records.append({
        "Superclass": cls_name,
        "AUC-ROC": round(auc_c, 4),
        "Sensitivity": f"{sens_c * 100:.2f}%",
        "Specificity": f"{spec_c * 100:.2f}%",
        "F1-Score": round(f1_c, 4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    })

eval_df = pd.DataFrame(metrics_records)

# Macro Summary
macro_auc = roc_auc_score(y_test, test_preds, average="macro")
binary_preds = np.zeros_like(test_preds)
for i, cls in enumerate(DIAGNOSTIC_SUPERCLASSES):
    binary_preds[:, i] = (test_preds[:, i] >= optimal_thresholds[cls]).astype(int)
macro_f1 = f1_score(y_test, binary_preds, average="macro")

print("=" * 80)
print(f"PTB-XL FOLD 10 HELD-OUT TEST EVALUATION MATRIX")
print(f"Overall Macro-AUROC : {macro_auc:.4f}")
print(f"Overall Macro-F1    : {macro_f1:.4f}")
print("=" * 80)
eval_df


In [ ]:
# ---------------------------------------------------------------------------
# 13. Multi-Class ROC Curves
# ---------------------------------------------------------------------------
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
colors = ['#2563eb', '#dc2626', '#16a34a', '#d97706', '#9333ea']

for idx, (cls_name, color) in enumerate(zip(DIAGNOSTIC_SUPERCLASSES, colors)):
    fpr, tpr, _ = roc_curve(y_test[:, idx], test_preds[:, idx])
    auc_val = roc_auc_score(y_test[:, idx], test_preds[:, idx])
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{cls_name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label="Chance")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
plt.ylabel("True Positive Rate (Sensitivity)", fontsize=11)
plt.title("PTB-XL Fold 10 Hold-Out ROC Curves", fontsize=13, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 14. Export to ONNX for CPU Microservice Deployment
# ---------------------------------------------------------------------------
import tf2onnx
import onnx
import onnxruntime as ort

onnx_path = os.path.join(OUTPUT_DIR, "resnet1d_ptbxl.onnx")

spec = (tf.TensorSpec((None, SIGNAL_LENGTH, NUM_LEADS), tf.float32, name="ecg_12lead_input"),)
model_proto, _ = tf2onnx.convert.from_keras(best_model, input_signature=spec, output_path=onnx_path)

print(f"✓ Successfully exported ONNX model to: {onnx_path}")
print(f"File size: {os.path.getsize(onnx_path) / (1024 * 1024):.2f} MB")

# Verify ONNX Runtime inference latency
session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dummy_input = np.random.randn(1, SIGNAL_LENGTH, NUM_LEADS).astype(np.float32)

# Benchmark 50 inference passes
latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    ort_outs = session.run(None, {"ecg_12lead_input": dummy_input})
    latencies.append((time.perf_counter() - t0) * 1000)

print(f"✓ ONNX CPU Inference Latency: {np.mean(latencies):.2f} ms ± {np.std(latencies):.2f} ms")
print(f"Output probability vector shape: {ort_outs[0].shape}")
